In [9]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    if root_dir.parts[-1:] == ('pollen',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Added the following directory to the PYTHONPATH: /Users/hongjiang/git/mlfs-book
HopsworksSettings initialized!


In [27]:
import datetime
import pandas as pd
from xgboost import XGBRegressor
import hopsworks
import json
from mlfs.airquality import util

today = datetime.datetime.now()
tomorrow = today + datetime.timedelta(days=1)

In [30]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store()

secrets = hopsworks.get_secrets_api()
location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
city = location['city']
latitude = location['latitude']
longitude = location['longitude']


2025-12-29 21:56:27,092 INFO: Closing external client and cleaning up certificates.
2025-12-29 21:56:27,094 INFO: Connection closed.
2025-12-29 21:56:27,095 INFO: Initializing external client
2025-12-29 21:56:27,096 INFO: Base URL: https://c.app.hopsworks.ai:443


2025-12-29 21:56:28,388 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1292436


In [18]:
import os
import joblib
import xgboost as xgb

# 1. 获取模型
mr = project.get_model_registry()
retrieved_model = mr.get_model(name="grass_pollen_model", version=3)

# 2. 下载
saved_model_dir = retrieved_model.download()
print(f"📦 模型已下载到: {saved_model_dir}")

# 3. 自动探测并加载模型
file_list = os.listdir(saved_model_dir)
print(f"包含文件: {file_list}")

if "pollen_model.pkl" in file_list:
    model_path = os.path.join(saved_model_dir, "pollen_model.pkl")
    model = joblib.load(model_path)
    print("✅ 成功加载 .pkl 模型")
elif "model.json" in file_list:
    model_path = os.path.join(saved_model_dir, "model.json")
    model = xgb.XGBRegressor()
    model.load_model(model_path)
    print("✅ 成功加载 .json 模型")

Downloading: 0.000%|          | 0/195267 elapsed<00:00 remaining<?

📦 模型已下载到: /var/folders/vv/h58bs1s95plbyzd7qh_6vd1c0000gn/T/17168353-42fe-4b17-a6cd-b0ea34fca31f/grass_pollen_model/3
包含文件: ['pollen_model.pkl']
✅ 成功加载 .pkl 模型


In [32]:
hourly_df = util.get_hourly_weather_forecast(city, latitude, longitude)
hourly_df = hourly_df.set_index('date')

# We will only make 1 daily prediction, so we will replace the hourly forecasts with a single daily forecast
# We only want the daily weather data, so only get weather at 12:00
daily_df = hourly_df.between_time('11:59', '12:01')
daily_df = daily_df.reset_index()
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.date
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['city'] = city
daily_df

Coordinates 59.25°N 18.0°E
Elevation 24.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s


,date,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city
0,2025-12-29,-1.60,0.0,23.686249,316.847595,Stockholm
1,2025-12-30,-1.85,0.0,24.627787,344.744812,Stockholm
2,2025-12-31,-4.05,0.0,2.811690,320.194489,Stockholm
3,2026-01-01,0.75,1.9,10.587917,162.181015,Stockholm
4,2026-01-02,-2.55,0.6,26.727423,27.255245,Stockholm
5,2026-01-03,-0.75,1.6,19.011953,18.778133,Stockholm
6,2026-01-04,-3.00,0.0,21.746504,6.654330,Stockholm


In [33]:





df_recent = daily_df


# 3. 在本地执行特征工程（不再依赖 weather_fg.read()）
df_recent = df_recent.sort_values('date')

# --- 计算时间特征 ---
df_recent['day_of_year'] = df_recent['date'].dt.dayofyear
df_recent['month'] = df_recent['date'].dt.month
df_recent['is_high_season'] = df_recent['day_of_year'].apply(lambda x: 1 if 140 <= x <= 250 else 0)

# --- 计算 GDD ---
T_base = 5.0
df_recent['gdd_daily'] = df_recent['temperature_2m_mean'].apply(lambda t: max(0, t - T_base))
df_recent['gdd_cumsum'] = df_recent.groupby(df_recent['date'].dt.year)['gdd_daily'].cumsum()

# --- 计算关键的 Lag 特征 ---
df_recent['precip_lag_1'] = df_recent['precipitation_sum'].shift(1)
df_recent['temp_lag_1'] = df_recent['temperature_2m_mean'].shift(1)
df_recent['wind_lag_1'] = df_recent['wind_speed_10m_max'].shift(1)

# 4. 筛选预测目标：只保留今天及之后的行，并删除第一行（因为第一行没有 lag 数据）
batch_data = df_recent[df_recent['date'] >= datetime.datetime.now().strftime('%Y-%m-%d')].dropna()



In [42]:
feature_cols = ['temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max', 'wind_direction_10m_dominant', 'city', 'day_of_year', 'month', 'is_high_season', 'gdd_daily', 'gdd_cumsum', 'precip_lag_1', 'temp_lag_1', 'wind_lag_1']
batch_data[feature_cols]
batch_data['predicted_pollen'] = model.predict(batch_data[feature_cols].drop(columns=['date_id', 'city'], errors='ignore'))
batch_data

,date,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city,day_of_year,month,is_high_season,gdd_daily,gdd_cumsum,precip_lag_1,temp_lag_1,wind_lag_1,predicted_pollen
1,2025-12-30,-1.85,0.0,24.627787,344.744812,Stockholm,364,12,0,0,0,0.0,-1.60,23.686249,-0.136340
2,2025-12-31,-4.05,0.0,2.811690,320.194489,Stockholm,365,12,0,0,0,0.0,-1.85,24.627787,-0.115114
3,2026-01-01,0.75,1.9,10.587917,162.181015,Stockholm,1,1,0,0,0,0.0,-4.05,2.811690,-0.023341
4,2026-01-02,-2.55,0.6,26.727423,27.255245,Stockholm,2,1,0,0,0,1.9,0.75,10.587917,0.006179
5,2026-01-03,-0.75,1.6,19.011953,18.778133,Stockholm,3,1,0,0,0,0.6,-2.55,26.727423,-0.004917
6,2026-01-04,-3.00,0.0,21.746504,6.654330,Stockholm,4,1,0,0,0,1.6,-0.75,19.011953,0.011683


In [43]:
batch_data['city'] = city
batch_data['days_before_forecast_day'] = range(1, len(batch_data)+1)
batch_data = batch_data.sort_values(by=['date'])
batch_data

,date,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city,day_of_year,month,is_high_season,gdd_daily,gdd_cumsum,precip_lag_1,temp_lag_1,wind_lag_1,predicted_pollen,days_before_forecast_day
1,2025-12-30,-1.85,0.0,24.627787,344.744812,Stockholm,364,12,0,0,0,0.0,-1.60,23.686249,-0.136340,1
2,2025-12-31,-4.05,0.0,2.811690,320.194489,Stockholm,365,12,0,0,0,0.0,-1.85,24.627787,-0.115114,2
3,2026-01-01,0.75,1.9,10.587917,162.181015,Stockholm,1,1,0,0,0,0.0,-4.05,2.811690,-0.023341,3
4,2026-01-02,-2.55,0.6,26.727423,27.255245,Stockholm,2,1,0,0,0,1.9,0.75,10.587917,0.006179,4
5,2026-01-03,-0.75,1.6,19.011953,18.778133,Stockholm,3,1,0,0,0,0.6,-2.55,26.727423,-0.004917,5
6,2026-01-04,-3.00,0.0,21.746504,6.654330,Stockholm,4,1,0,0,0,1.6,-0.75,19.011953,0.011683,6


In [44]:
monitor_fg = fs.get_or_create_feature_group(
    name='pollen_predictions',
    description='Pollen prediction monitoring',
    version=1,
    primary_key=['city','date','days_before_forecast_day'],
    event_time="date"
)

In [45]:
monitor_fg.insert(batch_data, wait=True)

Feature Group created successfully, explore it at 
https://c.app.hopsworks.ai:443/p/1292436/fs/1265790/fg/1880470


Uploading Dataframe: 100.00% |█| Rows 6/6 | Elapsed Time: 00:01 | Remaining Time


Launching job: pollen_predictions_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1292436/jobs/named/pollen_predictions_1_offline_fg_materialization/executions
2025-12-29 22:17:48,183 INFO: Waiting for execution to finish. Current state: INITIALIZING. Final status: UNDEFINED
2025-12-29 22:17:51,371 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2025-12-29 22:19:27,462 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2025-12-29 22:19:27,656 INFO: Waiting for log aggregation to finish.
2025-12-29 22:19:39,807 INFO: Execution finished successfully.


(Job('pollen_predictions_1_offline_fg_materialization', 'SPARK'), None)

In [47]:
pollen_fg = fs.get_feature_group(name='pollen', version=1)
pollen_df = pollen_fg.read()

outcome_df = pollen_df[['date', 'pollen_level']]
preds_df = monitoring_df[['date', 'predicted_pollen']]

hindcast_df = pd.merge(preds_df, outcome_df, on="date")
hindcast_df = hindcast_df.sort_values(by=['date'])
print(hindcast_df)

AttributeError: 'NoneType' object has no attribute 'read'